In [7]:
import logging
import time
import pandas as pd
from habanero import Crossref
cr = Crossref()
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type


cr = Crossref(mailto="m.n.khanji@umcg.nl")


retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([([ConnectionError, TimeoutError])])
)()

logging.basicConfig(
    filename='DOI.log',
    filemode='w',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

start_time = time.time()

csv_file = 'pmid_journal_doi_issn.csv'       #get csv file
df = pd.read_csv(csv_file)
dois = df.iloc[:, 2].tolist() # get dois from column (3rd in this case)

def publisher_crossref_doi(dois):
    publishers = []
    for doi in dois:
        try:
            work = cr.works(ids=doi)
            publisher = work["message"].get("publisher")
            publishers.append(publisher)
        except Exception as e:
            logging.error(f"Error: API request failed for {doi}: {e}")
            publishers.append(None)
    return publishers

# Call function and assign result to a list
publisher_list = publisher_crossref_doi(dois)

# Add the publisher_list to the DataFrame as a new column
df['publisher'] = publisher_list

# Save the updated DataFrame to a new CSV file
df.to_csv('pmid_doi_journal_issn_publisher.csv', index=False)
logging.info("Done")

In [8]:
# Load the CSV file
nan = pd.read_csv('pmid_doi_journal_issn_publisher.csv')

# Identify the column with missing values
publisher = 'publisher'

# Find the missing values in the column
missing_values = nan[publisher].isnull()

# Create a new DataFrame with only the rows that have missing values
missing_df = nan[missing_values]
print(missing_df)

         pmid                        journal  doi       issn  publisher
0    15588140                    Physiol Res  NaN  0862-8408        NaN
1    17487399                      Oncol Rep  NaN  1021-335X        NaN
2    12894873  Arch Immunol Ther Exp (Warsz)  NaN  0004-069X        NaN
3    25593506                        Mol Vis  NaN  1090-0535        NaN
4    18837167        Folia Microbiol (Praha)  NaN  0015-5632        NaN
..        ...                            ...  ...        ...        ...
672  18468426              Pol Arch Med Wewn  NaN        NaN        NaN
673  12820370                 Anticancer Res  NaN  0250-7005        NaN
674  12593473                       Clin Lab  NaN  1433-6510        NaN
675  18304426  Zhonghua Gan Zang Bing Za Zhi  NaN  1007-3418        NaN
676  20608180                Mol Biol (Mosk)  NaN  0026-8984        NaN

[677 rows x 5 columns]


In [ ]:

def get_publisher_ids_from_issn(missing_df):
    """
    Read ISSNs from a the missing_df dataframe, query the CrossRef API, and return a new DataFrame with publisher IDs.
    """
    # Extract ISSNs from 2nd column of the csv file
    issns = missing_df.iloc[:, 3].tolist() 

    publisher_ids = []

    # Iterate over the list of ISSNs
    for issn in issns:
        logging.info(f"Processing ISSN: {issn}")

        # Define the base URL with ISSN and select parameters
        url = f"https://api.crossref.org/works?filter=issn:{issn}&select=publisher"

        # Make GET request to the CrossRef API
        response = requests.get(url)

        # Check for successful response (status code 200)
        if response.status_code == 200:

            # Parse JSON response
            data = json.loads(response.text)
            if "message" in data:
                if "items" in data["message"] and data["message"]["items"]:
                    if len(data["message"]["items"]) > 0:
                        first_item = data["message"]["items"][0]
                        if isinstance(first_item, dict):  # Check if first_item is a dictionary
                            for key, value in first_item.items():
                                publisher_ids.append(value)  # Append the value to the list
                            else:
                                publisher_ids.append(str(first_item))  # Convert the list to a string and append it
                    else:
                        # Handle failed API request
                        logging.error(f"Error: API request failed {response.status_code} for {issn}")
                        publisher_ids.append(None)  # Append None if the API request fails

    # Create a new DataFrame with the original DataFrame adding publisher_ids list
    new_df = pd.DataFrame(list(zip(missing_df.values.tolist(), publisher_ids)), columns=['PMID', 'doi', 'issn', 'publisher'])
    return new_df

new_df = get_publisher_ids_from_issn(missing_df)

# Export the new_df to a new CSV file called PMID_Publisher.csv
new_df.to_csv('publishers_FINAL.csv', index=False)
